In [ ]:
from pathlib import Path
import csv
import json
import re

In [129]:
def csv_to_survey(rows, page_name="page1", page_title="Discovery", page_description="Phase 1 of your project."):
    """Translate CSV schema rows into a SurveyJS-compatible survey definition."""
    elements = []
    text_or_suggestion_groups = set()

    for row in rows:
        reference = row["Reference"].strip()
        field_text = row["Field text"].strip()
        component_type = row["Component type"].strip().lower()
        trigger = row["Trigger"].strip()
        if trigger.lower() == "none":
            trigger = ""

        if not reference or not field_text:
            continue

        group_match = re.match(r"\d+", reference)
        field_group = group_match.group() if group_match else None
        is_text_or_suggestion = "T" in reference or reference.endswith("S")
        line_break_needed = reference.isdigit()
        if is_text_or_suggestion and field_group not in text_or_suggestion_groups:
            line_break_needed = True
            text_or_suggestion_groups.add(field_group)

        element = {"type": {"radios": "radiogroup", "suggestion": "expression"}.get(component_type, component_type), "name": reference}
        element["startWithNewLine"] = line_break_needed

        if reference.endswith("S"):
            element["title"] = f"{reference[:-1]} {field_text}".rstrip()
        elif reference.find("T") != -1:
            element["title"] = field_text
        else:
            element["title"] = f"{reference}. {field_text}"

        trigger_parts = [part.strip() for part in re.split(r"\s+OR\s+", trigger, flags=re.IGNORECASE)]
        trigger_match = None
        if trigger_parts and trigger_parts[0]:
            trigger_match = re.fullmatch(r"(.+?)(<>|=|-)(.+)", trigger_parts[0])
            if trigger_match:
                question, operator, first_value = (part.strip() for part in trigger_match.groups())
                operator = "=" if operator == "-" else operator
                values = [first_value] + [part for part in trigger_parts[1:] if part]
                element["visibleIf"] = " or ".join(
                    f"{{{question}}} {operator} '{value}'" for value in values
                )

        if element["type"] == "radiogroup":
            choices = [
                row.get(option, "").strip()
                for option in ("Option a", "Option b", "Option c")
                if row.get(option, "").strip()
            ]
            element["choices"] = choices
            element["allowClear"] = True
            element["isRequired"] = True
            if trigger_match and operator == "=":
                element["resetValueIf"] = " and ".join(
                    f"{{{question}}} <> '{value}'" for value in values
                )

        elements.append(element)

    return {
        "pages": [{
            "name": page_name,
            "title": page_title,
            "description": page_description,
            "elements": elements,
        }],
        "headerView": "advanced",
    }

In [130]:
csv_path = Path("daief-schema-eg.csv")
survey_path = Path("survey.json")

with csv_path.open(newline="", encoding="utf-8-sig") as csv_file:
    schema = list(csv.DictReader(csv_file))

with survey_path.open(encoding="utf-8") as survey_file:
    survey = json.load(survey_file)

translated_survey = csv_to_survey(schema)

In [128]:
output_path = Path("translated-survey.json")
with output_path.open("w", encoding="utf-8") as output_file:
    json.dump(translated_survey, output_file, ensure_ascii=False, indent=2)
